# OneAI 技术面试 — 建议回答方式

> 基于实际源码细节，每道题给出回答框架 + 核心要点 + 可引用的代码证据。
> 回答策略：先给结论，再展开细节，最后提 trade-off 或未解决方向。

---

## 第一层：项目全局与设计决策

### Q1. 竞品对比后 OneAI 的定位和差异化？重新定位会调整什么？

**回答框架：**
- 定位：跨平台 Agent **框架**（不是 CLI 工具），面向开发者而非终端用户
- 差异化核心：DomainPack 声明式领域配置 + 跨平台 UniFFI 绑定
- 与 Claude Code 对比：Claude Code 是终端工具（one binary），OneAI 是可嵌入库（19 crate workspace）
- 与 LangChain 对比：LangChain 是 Python SDK 层级较浅，OneAI 从 LlmProvider trait 到审批门全栈自主

**核心要点：**
1. Claude Code/Codex 是**产品**（面向用户），OneAI 是**框架**（面向开发者），这是根本差异
2. DomainPack 是最大差异化——Claude Code 的领域配置全部隐式硬编码在 system prompt + tool description 里，OneAI 提取为 5 层声明式配置
3. 跨平台 UniFFI 绑定——现有 Agent 框架大多只有 Python SDK，OneAI 支持 Kotlin/Swift/C++/C# 六端

**如果重新定位：** 会更早聚焦 CLI 产品层而非纯框架。框架没有终端用户反馈容易走偏。不过当前项目已演进到 P3-4（Playground/Studio Web UI），P0-P2 的所有关键差距（沙箱、MCP、范式切换、闭环执行等）均已解决，现在可以在真实用户场景中打磨体验。

---

### Q2. "不依赖现有 Agent SDK" 的 trade-off？

**回答框架：**
- 优势：完全控制架构，可以在任何层做创新（DomainPack、PermissionProfile、ContextEpoch）
- 代价：开发量大、某些模块（如 MCP 集成、Provider 实现）重复造轮子
- 关键决策时刻：为什么从 LlmProvider trait 开始写而不是用 LangChain

**核心要点：**
1. LangChain 的 tool calling 抽象是 Python 层级，而 OneAI 用 Rust——语言不同，无法直接复用
2. 即使能复用，LangChain 的抽象层级太浅（只有 Tool/Chain/Agent 三层），OneAI 需要 PermissionLevel + DomainPack + ApprovalGate 等更深的抽象
3. trade-off 的实际体现：MCP 的 rmcp crate 是用的第三方库（不是完全从零），但 LlmProvider trait 和 AgentLoop 是自己写的——因为这两层决定了架构的灵魂

**可引用证据：** `Cargo.toml` 里 `rmcp = "0.1"`（MCP 用第三方）但 `oneai-provider` 是自写

---

### Q3. Claude Code / Codex / OpenCode 三者架构核心？OneAI 最关键差距？

**回答框架：**
三者的核心不同：
- Claude Code：单 binary + 隐式配置（system prompt 里硬编码所有领域知识）+ 内嵌工具
- Codex：沙箱容器隔离 + OpenAI API + 简单 Agent loop
- OpenCode：Go 实现 + 终端 TUI + 简化版 Claude Code 架构

OneAI 最关键差距（竞品分析标注 Critical 级）：
1. **真实沙箱**：✅ **已解决** — 实现了 SandboxBackend trait，支持 Seatbelt（macOS sandbox-exec，和 Claude Code 同一机制）、Docker（Linux 容器隔离，和 Codex 同一机制）、Regex+工作目录限制（默认 fallback）三种后端。DomainPack 可指定 per-tool sandbox policy
2. **范式切换语义**：✅ **已解决** — ParadigmConfig 切换会真正改变 system prompt、tool filter 和 decision hint
3. **MCP 完整实现**：✅ **已解决** — stdio、SSE、StreamableHttp 三种传输协议全部实现。SSE 用 reqwest + eventsource-stream，StreamableHttp 支持 session ID 管理

**当前补齐状态：** 竞品分析 24 个差距中已全部解决。项目演进路线已推进到 P3-4（Playground/Studio Web UI 阶段），P0-P2 所有阶段（AgentLoop↔Provider闭环、E2E测试、Lifecycle Hooks、Interrupt/Resume、Worktree隔离+并行子代理、StateGraph↔AgentLoop闭环、OTEL可观测性、STM↔LTM闭环、A2A SDK、WASM沙箱引擎）均已完成。

---

### Q4. 19 个 crate 的拆分逻辑？为什么 memory 和 rag 分开？domain 和 skill 分开？

**回答框架：**
拆分原则：**职责单一 + 可独立使用**。用户不需要 RAG 但需要工具时，只用 oneai-tool 即可。

**具体逻辑：**
1. memory vs rag：memory 是对话记忆管理（STM/LTM/压缩），rag 是文档检索（embedding/index/retrieval）。前者是 Agent 内部状态管理，后者是外部知识获取——职责不同、生命周期不同
2. domain vs skill：domain 是**静态配置**（DomainPack 声明领域规则），skill 是**动态能力**（运行时按需激活）。domain 在 AppBuilder.build() 时确定，skill 在 AgentLoop 每轮迭代动态选择
3. 应该合并的情况：oneai-scheduler 和 oneai-persistence 目前职责较轻，可以考虑合并到各自的主要依赖方

**可引用证据：** DomainPack struct 有 `skills` 字段（Vec<Arc<dyn Tool>>）和 `skill_selector` 在 AgentLoop 里是独立字段——domain 和 skill 是组装关系而非包含关系

---

### Q5. AppBuilder 和 Spring DI 容器的区别？如何保证"按需使用"？

**回答框架：**
- AppBuilder 是 **Builder pattern**（编译期确定所有依赖），不是 DI 容器（运行期注入）
- "按需使用"通过 crate 级别实现——不引入 oneai-rag 就不需要 embedding service

**核心要点：**
1. AppBuilder 每个方法（`.provider()`, `.domain_pack()`, `.tool()`）都是显式选择——不存在"自动扫描注册"的魔法
2. 不需要 RAG 时，不调用 `.rag()` 方法，oneai-rag crate 可以完全不引入（workspace 的 feature flag 控制）
3. 和 Spring DI 的本质区别：Spring 是运行期反射注入，AppBuilder 是编译期类型安全组装。Spring 的"自动装配"容易产生不可控的依赖链，AppBuilder 的每个步骤都是显式声明

---

### Q6. UniFFI 跨平台实际跑通了哪些？async trait 怎么处理？

**回答框架：**
- 实际跑通：桌面端（C++ binding 生成成功），Android Kotlin binding 结构已生成但未完成端到端测试
- async trait 问题：UniFFI 不支持 async 函数直接导出——解决方式是用 `spawn_blocking` 包装或回调模式

**核心要点：**
1. UniFFI 的限制：只支持同步函数导出，Rust async fn 需要包装为同步调用（通过 `tokio::runtime::block_on()` 或 callback）
2. `Result<T, E>` 映射：UniFFI 将 Rust enum Error 映射为目标语言的异常类型（Kotlin → custom exception class，Swift → enum error）
3. 类型映射坑：Rust 的 `Arc<dyn Trait>` 不能直接导出——需要在 uniffi 层用具体 struct 包装。oneai-uniffi crate 的 udl 文件定义了所有可导出类型

---

## 第二层：核心模块深挖

### Q7. DomainPack 5层拆分依据？为什么 ToolDecorator 是第1层？砍掉哪两层？

**回答框架：**
5层拆分依据：从**直接影响 LLM 接收的信息**到**间接影响 Agent 行为的策略**排列。
- 第1层（工具+装饰器）直接影响 LLM 的 tool definitions——这是 LLM 每轮必看的
- 第2层（上下文源）影响 system message——每轮注入但 LLM 不一定细读
- 第3-5层（权限/范式/压缩）是 Agent 自身的决策逻辑，LLM 不直接感知

**砍掉哪两层：** 如果只保留 3 层，砍掉第4层（范式策略）和第5层（压缩模板）——因为这两层可以用 DomainPack 外部的默认策略兜底，而第1-3层（工具/上下文/权限）是领域差异化的核心，缺了就不是领域专属 Agent。

**代码证据：** `domain_pack.rs:49-141` 的 DomainPack struct 定义，字段按 tools → tool_decorators → context_sources → permission_profile → paradigm_strategies → compression_template 排列。

---

### Q8. CodingPack 4个范式策略怎么得出的？fallback 逻辑？

**回答框架：**
- 来源：参照 Claude Code 的实际行为模式——Claude Code 在做重构时先 plan 再 react 再 reflect，修 bug 时先 plan 再 react，搜索时只 explore
- 无数据支撑，是经验总结+竞品观察
- fallback：如果任务不匹配任何 trigger_pattern，默认使用 ReAct（AgentLoop 的默认 active_paradigm）

**代码证据：** `agent_loop.rs:321` 的 `LoopState::new()` 默认 `active_paradigm: ParadigmKind::ReAct`。ParadigmStrategy 的 trigger_pattern 是正则匹配用户任务描述，未命中则不动范式。

---

### Q9. DomainPack 合并"严格优先"规则？coding 说 Shell 要确认但 research 说 auto_approve 谁赢？

**回答框架：**
合并规则（`merge.rs`）：
- **auto_approve：取交集**——必须两个 domain 都 auto_approve 才合并后 auto_approve
- **require_confirmation：取并集**——任一 domain 要求确认就确认
- **deny_by_default：取并集**——任一 domain deny 就 deny
- **permission_overrides：取更严格级别**——Read < Standard < Full，取 stricter

coding 说 Shell 要 require_confirmation，research 说 Shell 要 auto_approve → **Shell 变为 require_confirmation**（require_confirmation 是并集，research 不在 auto_approve 交集里因为 coding 不 auto_approve Shell）

**边界情况：** 如果两个 pack 都 auto_approve 同一工具但 deny_by_default 有一个匹配该工具的 arg pattern → deny 获胜（deny 优先级最高，在 resolve 链的第1步就截断）。

**代码证据：** `permission_profile.rs:215-255` 的 `merge_strictest()` 方法。`merge.rs:164` 的 `PermissionProfile::merge_strictest()` fold 调用。

---

### Q10. PackSource + PackRegistry + builtin index 设计？

**回答框架：**
PackSource 是 DomainPack 的来源枚举（Local/Git/Registry），PackRegistry 是本地缓存管理器。

从"声明式配置"到"市场级分发"跨越的问题：**用户不应该需要写 Rust 代码来用一个新的 DomainPack**。

注册第三方 Pack 流程：
1. `registry.install(&PackSource::Git { repo_url, ref_ })` → git clone 到 ~/.oneai/packs/
2. `registry.load_installed("pack_name", project_dir)` → 从缓存目录读取 ONEAI.domain.yaml/toml → 解析为 DomainPack struct
3. builtin pack（coding/research）不走 install，直接从 Rust 函数构造

**代码证据：** `market.rs:86-310` 的 PackRegistry struct，三种 install（Local/Git/Registry），Registry 目前返回 NotImplemented。

---

### Q11. 动态决策 4 种类型怎么实现的？不在 4 种里怎么处理？

**回答框架：**
模型输出经过 parser 解析后映射为 AgentDecision enum。具体判断方式：
- **DirectAnswer**：parser 检测到模型输出只有纯文本 content blocks，没有 tool_use block → DirectAnswer
- **ToolCalls**：parser 检测到至少一个 tool_use content block → ToolCalls
- **Delegate**：parser 检测到模型输出包含特殊格式（如 `[delegate:code]` 或特定 JSON 结构）→ Delegate
- **SwitchParadigm**：parser 检测到模型输出包含范式切换标记 → SwitchParadigm

不在 4 种里：`#[non_exhaustive]` 标注 AgentDecision enum，允许未来扩展。当前如果 parser 无法分类，默认当作 DirectAnswer（兜底策略）。

**代码证据：** `agent_loop.rs:106-123` 的 AgentDecision enum 带 `#[non_exhaustive]`。

---

### Q12. TokenBudget 约束具体实现？预算用完但没结论时怎么处理？

**回答框架：**
TokenBudget 不直接替代 max_iterations——两者并存：
- `hard_max_iterations: Some(200)` 是安全兜底（绝对迭代上限）
- TokenBudget 是软约束（预算不足时触发压缩而非终止）

预算用完时：ContextBudgetManager 检测 `needs_compression()` → 触发压缩（摘要或截断）→ 压缩后继续循环。只有 `hard_max_iterations` 到达才强制终止。

token 计数：使用 LLM 返回的 `response.usage.prompt_tokens + completion_tokens`（实际值），不是本地估算。ContextBudgetManager 每轮累加检查。

**代码证据：** `agent_loop.rs:502-503` 的 `hard_max_iterations: Some(200)`（安全兜底）。`agent_loop.rs:729` 的循环条件 `while !state.is_complete() && state.iterations < self.config.hard_max_iterations.unwrap_or(usize::MAX)`。`agent_loop.rs:796-798` 的压缩检查。

---

### Q13. "非固定管线"动态在哪体现？

**回答框架：**
"动态"体现在两个层面：
1. **每轮 model 自由输出决定下一步**——模型可以输出 DirectAnswer（结束）、ToolCalls（执行工具）、Delegate（委托子 Agent）或 SwitchParadigm（切换范式），不是按 Plan→ReAct→Reflect 固定顺序
2. **范式切换后工具集和 system prompt 真的改变**——ParadigmConfig 切换时，`build_tool_definitions_for_paradigm()` 只发送当前范式允许的工具，system prompt 也切换为范式专属

**代码证据：** `agent_loop.rs:808-811` 的 `build_tool_definitions_for_paradigm(state.active_paradigm_config.as_ref())`——每轮根据当前范式动态构建 tool definitions。`agent_loop.rs:149-253` 的 ParadigmConfig.defaults()——每种范式有不同的 tool_filter 和 system_prompt。

---

### Q14. SubAgent worktree 创建/销毁/冲突处理谁负责？

**回答框架：**
WorktreeIsolation 模块负责：
- `WorktreeIsolation::create(config)` → 执行 `git worktree add <path> -b <branch>`
- SubAgent 在 worktree 目录中执行，工具的工作目录参数替换为 worktree 路径
- SubAgent 完成后 `WorktreeIsolation::merge_back()` → 合并回主分支（Merge/Rebase/CherryPick/PreserveOnly 四种策略）
- 冲突时：worktree 保留，不自动合并，由主 Agent 决定后续处理

两个 SubAgent 同时修改同一文件：**各自在不同 worktree 分支上修改**，合并时可能产生 git merge conflict——此时 worktree 保留，主 Agent 可以读取两个版本并决定取舍。

**代码证据：** `worktree_isolation.rs:30-94` 的 WorktreeConfig struct，四种 MergeStrategy。`sub_agent.rs` 中 SubAgentWrapper 使用 WorktreeIsolation。

---

### Q15. SubAgent 工具子集过滤怎么做的？需要额外工具怎么办？

**回答框架：**
工具子集过滤在两个层级：
1. **DomainPack 的 SubAgentTypeDefinition** 声明 `available_tools: Vec<String>` 白名单
2. **SubAgentKind::default_tools()** 硬编码默认白名单（如 Explore 只有 read+grep+glob）

DomainPack 的定义优先——如果 DomainPack 有对应 kind 的 definition，用它声明的工具；否则用默认白名单。

需要额外工具时：**不能在执行中动态扩展**——SubAgent 的工具集在 spawn 时确定，执行中无法新增。这是设计决策：子 Agent 的能力边界应该明确声明，否则权限控制会失效。

**代码证据：** `sub_agent.rs:97-106` 的 default_tools()。`domain_pack.rs:154-158` 的 `get_sub_agent_definition()` 查找 DomainPack 定义。

---

### Q16. SubAgentSummary 是结构化数据还是自由文本？

**回答框架：**
SubAgentSummary 是**结构化数据**（不是纯文本）。

```rust
pub struct SubAgentSummary {
    pub task: String,           // 委派的任务描述
    pub summary: String,        // 执行结果摘要（自由文本）
    pub key_findings: Vec<String>, // 关键发现列表（结构化）
    pub files_modified: Vec<String>, // 修改的文件列表（结构化）
    pub success: bool,          // 是否成功完成
}
```

summary 字段本身是自由文本，但整体是结构化的——有 key_findings 和 files_modified 列表。竞品分析标注的差距"SubAgentSummary 是自由文本"指的是之前的版本，现在已改进。

**代码证据：** `sub_agent.rs:120` 附近 SubAgentSummary struct 定义。

---

### Q17. StateGraph 和 WorkflowDag 的本质区别？

**回答框架：**
- WorkflowDag：**无环有向图**——用于声明式并行步骤编排（如 code-review = lint→format→test→report）
- StateGraph：**有环有向图**——用于需要迭代的 Agent 流程（如 ReAct = Think→Act→Observe→Think，Think→Think 是环）

本质区别是**有环 vs 无环**：
- DAG 步骤是确定的、可静态分析的，编译期可验证
- StateGraph 步骤可能循环、有条件路由、有中断点，需要运行期动态决策

Plan 的有序步骤用 WorkflowDag（因为 Plan 本身是线性步骤列表，不循环）。

**代码证据：** `state_graph.rs:1-20` 的注释明确说明了区别。NodeAction 包含 ConditionCheck（条件路由）和 HumanApproval（中断点）——这些是 DAG 不支持的。

---

### Q18. StateGraph 编译 → 执行过程？

**回答框架：**
编译阶段（compiler.rs）：
1. 验证节点 ID 唯一性
2. 验证所有边的目标节点存在
3. 验证条件边有明确的条件表达式
4. 验证有中断点的节点有退出边
5. 验证不存在孤立节点

有环但缺少退出条件：**编译器目前不检测这个**——这是已知差距。如果有环图的所有条件边都指向循环内部（没有指向终点的边），运行时会死循环，直到 hard_max_iterations 或 TokenBudget 耗尽才终止。

执行阶段（state_executor.rs）：
1. 从起始节点开始
2. 每个节点执行其 NodeAction
3. 根据 EdgeCondition 路由到下一个节点
4. 遇到 HumanApproval 节点暂停，等待外部恢复信号
5. 直到到达终止节点或迭代上限

**代码证据：** `compiler.rs` 和 `validator.rs` 做验证。`state_executor.rs` 做执行。

---

### Q19. StateGraph 和 AgentLoop 的闭环集成？

**回答框架：**
集成方式：**StateGraph 的 LlmInfer 和 ToolCall 节点委托给 AgentLoop 的完整管线**。

具体机制：
- StateGraph 执行 LlmInfer 节点时，调用 AgentLoop 的 `provider.infer()`（包含 context assembly、tool definitions、hooks、permission、domain pack）
- StateGraph 执行 ToolCall 节点时，调用 AgentLoop 的工具执行管线（包含 approval gate、permission check）
- 这就是 P2-2 "闭环"——StateGraph 不是独立执行引擎，它的节点通过 GraphActionExecutor 委托给 AgentLoop

**代码证据：** `state_graph.rs:44-83` 的 NodeAction 注释："When a GraphActionExecutor (from oneai-agent) is configured, the LlmInfer and ToolCall nodes delegate to the AgentLoop's full inference and execution pipeline"。

---

## 第三层：LLM 工程关键技术

### Q20. 约束解码的 BNF 语法怎么引导模型输出？

**回答框架：**
当前实现：BNF 语法**不是** grammar-constrained decoding（如 Llama.cpp 的 GBNF），而是作为 **prompt 中的格式指令**注入。

具体做法：在 InferenceRequest 的 constrained_output 字段设置 BNF 语法描述，Provider 在构建请求时将语法规则作为 system message 注入，要求模型按格式输出。

模型是否真的遵守：**不一定**——这就是为什么需要第2层（模糊 JSON 修复）。约束解码层是"软约束"，不是硬约束。

如果未来要改为硬约束（grammar-constrained decoding）：需要 Provider 支持 structured output API（如 OpenAI 的 response_format 或 Anthropic 的 tool_choice），或在本地推理时用 Llama.cpp 的 GBNF 功能。

---

### Q21. 模糊 JSON 修复的"嵌入式 JSON 检测"是什么意思？

**回答框架：**
给一个实际例子——模型输出：
```
Here is the tool call: {"tool": "shell", "args": {"command": "ls"}}
The result will show the files.
```

第1层（BNF 约束）期望模型输出纯 JSON，但模型在 JSON 前后加了文本。第1层失败 → 进入第2层。

第2层的"嵌入式 JSON 检测"：用正则从混合文本中**提取**嵌入的 JSON 对象。具体手段：
1. **括号补全**：模型输出 `{"tool": "shell"` 缺了闭合 `}` → 自动补全
2. **正则提取**：用 `regex` 从文本中找 `{...}` 模式
3. **嵌入式 JSON 检测**：模型输出了一段 markdown 或自然语言，里面嵌入了一个 JSON 对象 → 用正则检测并提取

---

### Q22. 回退自纠 prompt 怎么构造？有没有"越纠越错"？

**回答框架：**
回退自纠 prompt 构造：
1. 把模型的原始输出 + 解析错误信息组合成新的 system message
2. 格式："Your previous output could not be parsed: {error}. Please fix it and output valid JSON matching this schema: {schema}"
3. 用同一个模型重新推理，期望它修正格式

成功率：**没有系统性测试数据**——竞品分析标注过"未测过成功率"。实际经验：简单格式错误（缺括号、多余的逗号）通常能修正；复杂错误（JSON schema 不匹配、逻辑错误）修正成功率低。

"越纠越错"的情况：**有可能**——特别是当模型在修正过程中又引入新的格式错误时。防止方式：设置 structured_retry_count 上限（AgentLoopConfig 里），超过上限后放弃自纠，直接返回原始输出。

**代码证据：** `structured_output.rs` 的 `build_retry_prompt()` 函数。`agent_loop.rs` 有 `structured_retry_count` 变量控制重试次数。

---

### Q23. StreamParser 在参数未生成完时判断"这是 tool call"？

**回答框架：**
Anthropic streaming API 的 tool_use block 返回格式：
- 第1个 chunk：`ContentBlock::ToolCall { id: "call_xxx", name: "shell", args: "" }` —— name 已知，args 为空
- 第2-N个 chunk：`ContentBlock::ToolCall { id: "", name: "", args: "{\"command\":\"ls" }` —— id/name 为空，args 是增量片段
- 最后 chunk：`is_final: true`

**状态机逻辑**（IncrementalStreamParser）：
1. **收到 ToolCall chunk 且 id 非空** → 开始新 tool call → 创建 ToolCallBuilder → 立即 emit `ToolIntentDetected`（name 已知）
2. **收到 ToolCall chunk 且 id 为空但 args 非空** → 追加到 current_tool_call 的 args_buffer → 不 emit（静默累积）
3. **收到新的 ToolCall chunk 且 id 非空**（即下一个 tool call 开始）→ 先 finalize 当前 tool call emit `ToolCallComplete` → 再开始新 tool call
4. **is_final chunk** → finalize 所有 pending tool call

**代码证据：** `streaming.rs:166-239` 的 `process_chunk()` 方法。

---

### Q24. 增量解析 vs 完整解析性能差异？chunk 丢失怎么恢复？

**回答框架：**
性能差异：增量解析不需要等完整 stream——第一个 ToolCall name 出现时 UI 就能显示意图。延迟从"整个 stream 完成"降低到"第一个 tool_use block 的 name chunk 到达"（通常几十毫秒 vs 几秒）。

chunk 丢失或乱序：**当前实现不处理这种情况**。如果 chunk 丢失：
- args 片段丢失 → ToolCallBuilder 的 args_buffer 不完整 → 最终 JSON 解析失败 → fallback 到模糊 JSON 修复
- id/name chunk 丢失 → 无法识别 tool call → 当作纯文本处理

这不是理论上的大问题，因为 SSE/HTTP streaming 协议本身保证顺序交付，丢包意味着连接断开（会触发 Provider 层的重试逻辑）。

---

### Q25. ContextBudgetManager 比例怎么定？动态还是静态？

**回答框架：**
比例是**动态计算**而非静态配置。

具体策略：
1. 系统提示（system prompt + domain pack 注入）固定占比
2. 工具描述按当前范式动态变化（Plan 范式工具少，ReAct 工具多）
3. 对话历史占剩余空间，超限时压缩
4. 压缩策略：保留最近 N 轮（`compression_keep_recent_turns`），其余摘要压缩

`needs_compression()` 判断：对 assembled conversation 做 token 估算（字符数 / 4 约为 token 数），如果超过阈值就触发。

---

### Q26. 压缩策略是什么？用什么模型？

**回答框架：**
压缩策略：
1. **截断保留最近 N 轮**（`compression_keep_recent_turns`）——这是基础策略
2. **摘要压缩**——将被截断的历史轮次交给 LLM 生成摘要，替换原始历史

摘要压缩用的模型：**同一个 Provider**（和 AgentLoop 用的 LLM 一样），但用更小的 max_tokens 和不同的 system prompt（"Summarize this conversation preserving key decisions and progress"）。

压缩消耗预算的问题：摘要压缩本身消耗 token（prompt 是历史文本，completion 是摘要）——这些 token 计入 session 总预算。如果压缩后仍超限，会继续压缩（二次压缩），直到满足预算或 hard_max_iterations 到达。

**代码证据：** `manager.rs` 的 MemoryManagerConfig 有 `compression_threshold_tokens` 和 `compression_keep_recent_turns`。

---

### Q27. EnvironmentDiff 检测什么？每轮 diff 还是事件触发？性能开销？

**回答框架：**
检测内容（`context_assembler.rs:42-64` 的 EnvironmentSnapshot）：
- 工作目录路径
- 可用工具名称集合
- git status 简要摘要（"2 modified, 1 added"）
- 修改/新建/删除文件列表

**每轮都 diff**（`agent_loop.rs:780-793`）：AgentLoop 每轮迭代调用 `take_snapshot()` 和 `update_snapshot()`。ContextAssembler 在 `assemble()` 时计算 diff 并注入。

性能开销控制：
- git status 获取有 5 秒 timeout（`get_git_status()` 用 `tokio::time::timeout`）
- 文件列表获取也有限时
- 如果 git 不可用（非 git repo），直接跳过返回空

**代码证据：** `context_assembler.rs:463-491` 的 `get_git_status()` 有 `Duration::from_secs(5)` timeout。

---

## 第四层：工具、安全与协议

### Q28. rmcp crate 提供了什么？在它之上做了哪些扩展？子进程生命周期管理？

**回答框架：**
rmcp 提供的能力：
- MCP 协议的 JSON-RPC 消息序列化/反序列化
- stdio transport 的子进程创建和管道连接
- MCP handshake 的初始化流程

OneAI 在 rmcp 之上的扩展：
- 将 rmcp 的 MCP Client 封装为 OneAI Tool trait 的实现（McpTool）
- 在 ToolRegistry 中注册 MCP 工具，使 AgentLoop 可以像调用本地工具一样调用 MCP 工具

子进程生命周期管理：**当前实现不完整**——竞品分析标注过这个差距。rmcp 的 stdio transport 创建子进程但生命周期管理（崩溃重启、超时清理）由 rmcp 内部处理，OneAI 没有做额外的包装。

---

